# EDA dan Preprocessing Sentiment Analysis

## Tujuan
1. memuat dataset mentah
2. melakukan EDA awal
3. menerapkan preprocessing teks dengan gaya `mct-nlp`
4. membuat fitur tambahan untuk modeling
5. menyimpan data bersih ke folder `data/processed`

## Ringkasan Awal
- Jumlah baris data mentah: **732**
- Jumlah label sentimen unik: **191**
- Tidak ditemukan missing value pada data mentah
- Ditemukan **26** duplikasi pada kolom teks
- Distribusi agregasi sentimen 3 kelas:
  - positive: **460**
  - negative: **190**
  - neutral: **82**


## mct-nlp

Pola utama yang diadopsi dari referensi:
- fungsi preprocessing dibuat sederhana dan modular
- hasil pembersihan disimpan pada kolom `cleaned_text`
- preprocessing dilakukan secara rule-based
- dokumentasi dibuat edukatif dan mudah dipresentasikan

Untuk dataset ini, pipeline dibangun dengan pendekatan:
`lowercase -> hapus HTML -> hapus URL -> ekspansi kontraksi -> normalisasi hashtag -> hapus mention -> hapus non-alfabet -> rapikan spasi`


In [ ]:
from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")
pd.set_option("display.max_colwidth", 120)

BASE_DIR = Path.cwd().resolve().parent
RAW_CSV = BASE_DIR / "data" / "raw" / "sentimentdataset.csv"
PROCESSED_CSV = BASE_DIR / "data" / "processed" / "clean_data.csv"

print("File mentah:", RAW_CSV)
print("File output:", PROCESSED_CSV)


In [ ]:
CONTRACTION_MAP = {
    "won't": "will not",
    "can't": "cannot",
    "i'm": "i am",
    "ain't": "is not",
    "let's": "let us",
    "isn't": "is not",
    "it's": "it is",
    "i've": "i have",
    "doesn't": "does not",
    "didn't": "did not",
    "don't": "do not",
    "wasn't": "was not",
    "weren't": "were not",
    "couldn't": "could not",
    "shouldn't": "should not",
    "wouldn't": "would not",
    "haven't": "have not",
    "hasn't": "has not",
    "hadn't": "had not",
    "they're": "they are",
    "we're": "we are",
    "you're": "you are",
    "he's": "he is",
    "she's": "she is",
    "that's": "that is",
    "what's": "what is",
    "who's": "who is",
    "there's": "there is",
    "here's": "here is",
    "where's": "where is",
    "i'll": "i will",
    "you'll": "you will",
    "he'll": "he will",
    "she'll": "she will",
    "we'll": "we will",
    "they'll": "they will",
    "i'd": "i would",
    "you'd": "you would",
    "he'd": "he would",
    "she'd": "she would",
    "we'd": "we would",
    "they'd": "they would",
    "you've": "you have",
    "we've": "we have",
    "they've": "they have",
}

POSITIVE_LABELS = {
    "Acceptance","Accomplishment","Admiration","Adoration","Adrenaline","Adventure","Affection","Amazement","Amusement",
    "Anticipation","Appreciation","Arousal","ArtisticBurst","Awe","Blessed","Breakthrough","Calmness","Captivation",
    "Celebration","Celestial Wonder","Challenge","Charm","Colorful","Compassion","Compassionate","Confidence","Confident",
    "Connection","Contentment","Coziness","Creative Inspiration","Creativity","Culinary Adventure","CulinaryOdyssey",
    "Dazzle","Determination","DreamChaser","Ecstasy","Elation","Elegance","Empathetic","Empowerment","Enchantment",
    "Energy","Engagement","Enjoyment","Enthusiasm","Envisioning History","Euphoria","Excitement","Exploration",
    "FestiveJoy","Free-spirited","Freedom","Friendship","Fulfillment","Grandeur","Grateful","Gratitude","Happiness",
    "Happy","Harmony","Heartwarming","Hope","Hopeful","Hypnotic","Iconic","Imagination","Immersion","Inspired",
    "Inspiration","Intrigue","Journey","Joy","Joy in Baking","JoyfulReunion","Kind","Kindness","Love","Marvel",
    "Melodic","Mesmerizing","Mindfulness","Mischievous","Motivation","Nature's Beauty","Ocean's Freedom","Optimism",
    "Overjoyed","Playful","PlayfulJoy","Positive","Positivity","Pride","Proud","Radiance","Rejuvenation","Relief",
    "Renewed Effort","Resilience","Reverence","Romance","Runway Creativity","Satisfaction","Serenity","Solace","Spark",
    "Success","Sympathy","Tenderness","Thrill","Thrilling Journey","Touched","Tranquility","Triumph","Vibrancy",
    "Whimsy","Winter Magic","Wonder","Wonderment","Zest"
}

NEGATIVE_LABELS = {
    "Anger","Anxiety","Apprehensive","Bad","Betrayal","Bitter","Bitterness","Boredom","Darkness","Desolation","Despair",
    "Desperation","Devastated","Disappointed","Disappointment","Disgust","Dismissive","Embarrassed","EmotionalStorm",
    "Envious","Envy","Exhaustion","Fear","Fearful","Frustrated","Frustration","Grief","Hate","Heartache","Heartbreak",
    "Helplessness","Intimidation","Isolation","Jealous","Jealousy","Loneliness","Loss","LostLove","Melancholy",
    "Miscalculation","Negative","Numbness","Obstacle","Overwhelmed","Pressure","Regret","Resentment","Ruins","Sad",
    "Sadness","Shame","Solitude","Sorrow","Suffering","Suspense","Whispers of the Past","Yearning"
}

def expand_contractions(text: str) -> str:
    words = text.split()
    expanded = [CONTRACTION_MAP.get(word, word) for word in words]
    return " ".join(expanded)

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = expand_contractions(text)
    text = re.sub(r"#(\w+)", r" \1 ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def zscore(series: pd.Series) -> pd.Series:
    series = pd.to_numeric(series, errors="coerce")
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return pd.Series([0.0] * len(series), index=series.index, dtype="float64")
    return ((series - series.mean()) / std).astype("float64")

def map_sentiment_group(label: str) -> str:
    if label in POSITIVE_LABELS:
        return "positive"
    if label in NEGATIVE_LABELS:
        return "negative"
    return "neutral"


In [ ]:
print("Membaca dataset mentah")
df_raw = pd.read_csv(RAW_CSV, sep=";")
print("Shape data mentah:", df_raw.shape)
display(df_raw.head())


In [ ]:
print("Informasi dataset")
display(pd.DataFrame({
    "kolom": df_raw.columns,
    "dtype": df_raw.dtypes.astype(str).values
}))

print("Missing value tiap kolom")
display(df_raw.isna().sum().to_frame("missing_count"))

print("Top 15 label sentimen")
df_sentiment = (
    df_raw["Sentiment"]
    .astype(str)
    .str.strip()
    .value_counts()
    .head(15)
    .reset_index()
)
df_sentiment.columns = ["sentiment", "jumlah"]
display(df_sentiment)

plt.figure(figsize=(12, 6))
sns.barplot(data=df_sentiment, x="jumlah", y="sentiment", palette="viridis")
plt.title("Top 15 Distribusi Sentimen Asli")
plt.xlabel("Jumlah")
plt.ylabel("Sentimen")
plt.tight_layout()
plt.show()

print("Distribusi platform")
df_platform = (
    df_raw["Platform"]
    .astype(str)
    .str.strip()
    .value_counts()
    .reset_index()
)
df_platform.columns = ["platform", "jumlah"]
display(df_platform)

plt.figure(figsize=(8, 4))
sns.barplot(data=df_platform, x="platform", y="jumlah", palette="magma")
plt.title("Distribusi Platform")
plt.xlabel("Platform")
plt.ylabel("Jumlah")
plt.tight_layout()
plt.show()


In [ ]:
print("Memulai preprocessing")

df = df_raw.copy()
df.columns = [col.strip() for col in df.columns]

rename_map = {
    "Unnamed: 0.1": "record_id",
    "Unnamed: 0": "source_index",
    "Text": "text",
    "Sentiment": "sentiment",
    "Timestamp": "timestamp",
    "User": "user",
    "Platform": "platform",
    "Hashtags": "hashtags",
    "Retweets": "retweets",
    "Likes": "likes",
    "Country": "country",
    "Year": "year",
    "Month": "month",
    "Day": "day",
    "Hour": "hour",
}
df = df.rename(columns=rename_map)

object_columns = ["text", "sentiment", "user", "platform", "hashtags", "country", "timestamp"]
for col in object_columns:
    df[col] = df[col].astype(str).str.strip()

numeric_columns = ["record_id", "source_index", "retweets", "likes", "year", "month", "day", "hour"]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["timestamp_dt"] = pd.to_datetime(df["timestamp"], format="%d/%m/%Y %H:%M", errors="coerce")

df = df.dropna(subset=["text", "sentiment"]).reset_index(drop=True)
df["sentiment_group"] = df["sentiment"].apply(map_sentiment_group)
df["cleaned_text"] = df["text"].apply(clean_text)
df = df[df["cleaned_text"].str.len() > 0].reset_index(drop=True)

df["text_length_chars"] = df["text"].str.len()
df["text_length_words"] = df["text"].str.split().str.len()
df["cleaned_text_length_chars"] = df["cleaned_text"].str.len()
df["cleaned_text_length_words"] = df["cleaned_text"].str.split().str.len()
df["has_hashtag"] = df["hashtags"].str.contains(r"#", regex=True).astype(int)
df["hashtag_count"] = df["hashtags"].str.count(r"#")
df["engagement_total"] = df["retweets"].fillna(0) + df["likes"].fillna(0)

scale_targets = [
    "retweets",
    "likes",
    "engagement_total",
    "text_length_chars",
    "text_length_words",
    "cleaned_text_length_chars",
    "cleaned_text_length_words",
    "hashtag_count",
]

for col in scale_targets:
    df[f"{col}_scaled"] = zscore(df[col]).round(6)

ordered_columns = [
    "record_id","source_index","timestamp","timestamp_dt","year","month","day","hour",
    "user","platform","country","hashtags","has_hashtag","hashtag_count",
    "text","cleaned_text","text_length_chars","text_length_words",
    "cleaned_text_length_chars","cleaned_text_length_words",
    "sentiment","sentiment_group","retweets","likes","engagement_total",
    "retweets_scaled","likes_scaled","engagement_total_scaled",
    "text_length_chars_scaled","text_length_words_scaled",
    "cleaned_text_length_chars_scaled","cleaned_text_length_words_scaled",
    "hashtag_count_scaled"
]

df_clean = df[ordered_columns].copy()

print("Preprocessing selesai")
print("Shape data bersih:", df_clean.shape)
display(df_clean.head())


In [ ]:
print("Contoh sebelum dan sesudah preprocessing")
examples = (
    df_clean[["text", "cleaned_text", "sentiment", "sentiment_group"]]
    .sample(n=min(10, len(df_clean)), random_state=42)
    .reset_index(drop=True)
)
display(examples)


In [ ]:
print("Distribusi sentimen 3 kelas")
df_polarity = df_clean["sentiment_group"].value_counts().reset_index()
df_polarity.columns = ["sentiment_group", "jumlah"]
display(df_polarity)

plt.figure(figsize=(6, 4))
sns.barplot(data=df_polarity, x="sentiment_group", y="jumlah", palette="Set2")
plt.title("Distribusi Sentiment Group")
plt.xlabel("Kelompok Sentimen")
plt.ylabel("Jumlah")
plt.tight_layout()
plt.show()


In [ ]:
print("Korelasi fitur numerik utama")
corr_cols = [
    "retweets", "likes", "engagement_total",
    "text_length_chars", "text_length_words",
    "cleaned_text_length_chars", "cleaned_text_length_words",
    "hour", "day", "month", "year"
]
corr_matrix = df_clean[corr_cols].corr(numeric_only=True)
display(corr_matrix.round(3))

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Heatmap Korelasi Fitur Numerik")
plt.tight_layout()
plt.show()


In [ ]:
print("Menyimpan clean_data.csv")
PROCESSED_CSV.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(PROCESSED_CSV, index=False)
print("File berhasil disimpan:", PROCESSED_CSV)
